# Experiment 5 · Box-prompt sensitivity

How robust is each segmentation model to the quality of its box prompt? This experiment
perturbs the ground-truth oracle bounding box and measures how the Dice score (DSC)
degrades as the prompt gets worse. Five perturbation **modes** are applied at inference
time:

- **tight** — the exact GT box (no perturbation, the reference);
- **jitter10 / jitter25 / jitter50** — each box edge is randomly displaced by
  ±10% / ±25% / ±50% of the box size (graceful-degradation sweep — the box still sits on
  the nodule);
- **shifted / adversarial** — the box is pushed off the nodule (a conditioning control:
  a good model must produce a *wrong* mask, not a box-shaped one).

The sweep spans three regimes — **zero-shot foundation** (SAM ViT-H, SAM2, SAM3, MedSAM,
MedSAM-2), **few-shot LoRA** (`f=0.5`), and **box-conditioned CNNs / nnU-Net v2** — across
all four datasets (DDTI, TN3K, ThyroidXL, Stanford AIMI). For every (mode, image) the
perturbed box is deterministic and **seeded** (seed 42 / per-image offset), so every model
is scored on identical boxes. Each row also reports the trivial **box-fill floor** (DSC
obtained by predicting the perturbed box itself) as the reference for `ΔFloor`.


In [ ]:
# Move to the repository root (the directory that contains pyproject.toml) so that
# the `thyroidbench` package is importable and the relative data / results paths
# resolve correctly.
import os
from pathlib import Path

cwd = Path.cwd().resolve()
for candidate in [cwd, *cwd.parents]:
    if (candidate / "pyproject.toml").exists():
        os.chdir(candidate)
        break
print("Repository root:", Path.cwd())


In [ ]:
# Check the inputs this experiment needs before doing anything slow. A missing
# dataset here means an unrun (or unplaced) setup step, not a bug in the experiment.
from pathlib import Path

REQUIRED = ['ddti', 'tn3k', 'thyroidxl', 'stanford_aimi']
GATED = {'thyroidxl', 'stanford_aimi'}

missing = [d for d in REQUIRED
           if not (Path('data/processed') / d / 'images').is_dir()
           or not (Path('data/splits') / f'{d}_test.csv').exists()]
if missing:
    print('Missing preprocessed data or splits for:', ', '.join(missing))
    for d in missing:
        if d in GATED:
            print(f'  {d:14s} access-gated -> place your approved copy first, see '
                  f'00_setup/01_place_gated_datasets.ipynb')
        else:
            print(f'  {d:14s} open -> download it with 00_setup/00_get_open_datasets.ipynb')
    print('Then run 00_setup/02_preprocess.ipynb and 00_setup/03_make_splits.ipynb.')
    print('\nYou can still run this experiment on whichever datasets ARE present '
          'by restricting the --dataset argument below.')
else:
    print('All four datasets are preprocessed and split.')


## Prerequisites

This experiment is **inference-only** — it re-uses checkpoints trained by earlier
experiments and the published foundation weights. Nothing here is trained. You need:

- **Foundation-model weights** (zero-shot regime) under a git-ignored `pretrained_models/`
  directory at the repository root — the same files documented in `thyroidbench/models/`
  (`sam_vit_h_4b8939.pth`, `sam2.1_hiera_large.pt`, `medsam_vit_b.pth`,
  `MedSAM2_latest.pt`, and the SAM3 checkpoint).
- **Few-shot LoRA checkpoints** (LoRA regime) from Experiment 3, read from
  `experiments/exp3_fewshot_lora/results/fewshot_<model>_<dataset>_f0.5/fewshot_<model>_<dataset>_f0.5_best.pt`.
- **Box-conditioned traditional checkpoints** (CNN / nnU-Net regime) from Experiment 1:
  the U-Net / TransUNet `*_box_sensitivity.csv` files and the trained nnU-Net folder under
  `experiments/exp1_fullsupervised/traditional_boxcond/results/`.

The sweep writes its CSVs under `results/`; the **Tables + figure** section below reads
them back once the sweep has finished.


## Run

In [ ]:
# Full sweep (foundation side): 5 zero-shot + 4 LoRA models x 4 datasets, skipping any
# config whose output CSV already exists. Inference-only but the large test splits still
# take several hours on one GPU. Uncomment to launch.
# !bash experiments/exp5_boxsens/run_boxsens_all.sh

# Illustrative single config — zero-shot SAM2 on DDTI, all perturbation modes.
# run_boxsens.py args: --regime {zeroshot,lora} --model {sam,sam2,sam3,medsam,medsam2}
#                      --dataset {ddti,tn3k,thyroidxl,stanford_aimi}
#                      [--fraction 0.5] [--device cuda] [--modes tight jitter10 ...]
!python experiments/exp5_boxsens/run_boxsens.py \
    --regime zeroshot --model sam2 --dataset ddti

# LoRA f=0.5 variant (reads the Exp 3 checkpoint):
# !python experiments/exp5_boxsens/run_boxsens.py \
#     --regime lora --model sam2 --dataset ddti --fraction 0.5

# Box-conditioned nnU-Net v2 sweep for one dataset:
# !python experiments/exp5_boxsens/run_nnunet_boxsens.py --dataset ddti


## Aggregate

`aggregate_boxsens.py` collates every sweep CSV into one tidy table,
`results/boxsens_summary.csv`: one row per regime x model x dataset x LoRA fraction x
perturbation mode, carrying DSC, IoU, HD95 and the box-fill delta. That table is what the
reported box-sensitivity results are read from. The box-conditioned CNN rows are folded in
from the Experiment 1 results tree, so run Experiment 1 first if you want the full table
rather than the foundation half.

If no sweep fragments are present the aggregator leaves the shipped summary untouched,
so running this in a fresh clone will not blank it.


In [ ]:
!python experiments/exp5_boxsens/aggregate_boxsens.py


## Results

The sweep writes one CSV per configuration under `results/`. The cell below locates them, then
loads a representative one — zero-shot SAM2 on DDTI — showing mean DSC per perturbation
mode alongside the box-fill floor and `ΔFloor`. Note the flat-then-cliff shape: DSC holds
under mild jitter, then collapses as the box drifts off the nodule (shifted / adversarial),
while `ΔFloor` stays near zero for `shifted`/`adversarial`, confirming the model is not
merely painting the box.

In [ ]:
import pandas as pd

# `aggregate_boxsens.py` folds the sweep into a single box-sensitivity table
# (one row per regime x model x dataset x perturbation mode). The per-config
# sweep CSVs are regenerated locally by `run_boxsens.py` / `run_nnunet_boxsens.py`
# and re-collated with `aggregate_boxsens.py`.
summary_path = "experiments/exp5_boxsens/results/boxsens_summary.csv"
df = pd.read_csv(summary_path)
print(f"Loaded {len(df)} rows from {summary_path}")
print("regimes:", df["regime"].unique().tolist())
print("modes:  ", df["mode"].unique().tolist())

# Representative foundation-regime sweep: zero-shot SAM2 on DDTI, all modes.
rep = df[(df["regime"] == "zeroshot") & (df["model"] == "sam2") & (df["dataset"] == "ddti")]
display(rep[["regime", "model", "dataset", "mode", "n", "dsc", "iou", "hd95"]])
